In [0]:
orders = spark.read.table("revenue_operations.silver.orders")
customers = spark.read.table("revenue_operations.silver.customers")
order_items = spark.read.table("revenue_operations.silver.order_items")
products = spark.read.table("revenue_operations.silver.products")
sellers = spark.read.table("revenue_operations.silver.sellers")
geolocation = spark.read.table("revenue_operations.silver.geolocation")
product_category_translation = spark.read.table("revenue_operations.silver.product_category_translation")
payments = spark.read.table("revenue_operations.silver.payments")
reviews = spark.read.table("revenue_operations.silver.reviews")

#### Fact Orders
- Intended grain = One row represents a single order by a customer, and its order, deliver status, with price and freigh value of item purchased 
- source Silver tables = revenue_operations.silver.orders, revenue_operations.silver.payments, revenue_operations.silver.reviews
- columns selected = order_id, customer_id, order_status, purchase_date, order_purchase_timestamp, delivery_date, delivery_days, estimated_delivery_days, late_delivery_flag, estimated_delivery_date, total_payment_value, number_of_payment_records, average_review_score.
- calculated fields = purchase_date, delivery_date, delivery_days, estimated_delivery_days, late_delivery_flag, estimated_delivery_date, total_payment_value, number_of_payment_records, average_review_score.
- joins used = LEFT JOINs to join orders to payments, on order_id, and to join reviews, on order_id
- row count = 99441
- duplicate-key validation = No duplicate keys
- null check on key fields = NO null in key fields

In [0]:
%sql
-- Create a temp view 
CREATE OR REPLACE TEMP VIEW fact_orders AS
SELECT *
FROM (
    SELECT * EXCEPT(reviews_avg.order_id)
    FROM (
        SELECT orders.order_id, 
                customer_id, 
                order_status, 
                DATE(order_purchase_timestamp) AS purchase_date, 
                order_purchase_timestamp,
                DATE(order_delivered_customer_date) AS delivery_date, 
                DATEDIFF(order_delivered_customer_date, order_purchase_timestamp) AS delivery_days,
                DATEDIFF(order_estimated_delivery_date, order_purchase_timestamp) AS estimated_delivery_days,
                (CASE 
                    WHEN order_delivered_customer_date > order_estimated_delivery_date THEN "late" 
                    WHEN order_delivered_customer_date IS NULL THEN "not_delivered"
                    WHEN order_estimated_delivery_date IS NULL THEN "processing"
                    ELSE "on_time" END) AS late_delivery_flag,
                DATE(order_estimated_delivery_date) AS estimated_delivery_date, 
                payment_records.total_payment_value, payment_records.number_of_payment_records
        FROM revenue_operations.silver.orders
        LEFT JOIN (
                    SELECT order_id, SUM(payment_value) AS total_payment_value, COUNT(*) AS number_of_payment_records
                    FROM revenue_operations.silver.payments
                    GROUP BY order_id) AS payment_records
                    ON orders.order_id = payment_records.order_id) AS order_purchase
    LEFT JOIN (
                SELECT order_id, AVG(review_score) AS average_review_score
                FROM revenue_operations.silver.reviews
                GROUP BY order_id) AS reviews_avg        
    ON order_purchase.order_id = reviews_avg.order_id)



In [0]:
%sql
-- Duplicate Key validation
SELECT order_id, COUNT(*) 
FROM fact_orders
GROUP BY order_id
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT *
FROM fact_orders
WHERE order_id IS NULL OR customer_id IS NULL

In [0]:
# Query the temp view to get the data into a DataFrame
fact_orders = spark.sql("SELECT * FROM fact_orders")
display(fact_orders)
fact_orders.count()

In [0]:
fact_orders.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.gold.fact_orders")

#### Fact Order Items
- Intended grain = One row represents a single order with what product was ordered, which seller sold it, and its prices with freight value
- Source Silver tables = revenue_operations.silver.order_items, revenue_operations.silver.products, revenue_operations.silver.product_category_translation
- Columns selected = order_id, product_id, seller_id, quantity, product_subtotal, freight_total, line_total, product_category
- Calculated fields = quantity, product_subtotal, freight_total, line_total
- Joins used = LEFT JOINs to join order_items to products, on product_id, and to join product_category_translation, on product_category_name
- row count = 102425 (10,225 rows less than order_items table because we are counting quantity for the products that have been ordered multiple quantity of same product by same customer with the same order id.)
- duplicate-key validation = No duplicate keys
- null check on key fields = NO null in key fields

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW fact_order_items AS
(SELECT order_id, product_id, seller_id, quantity,
        product_subtotal, freight_total, (product_subtotal + freight_total) AS line_total, product_category_name_english AS product_category
FROM
    (SELECT * EXCEPT(products_quantity_category.source_file_name, 
                    product_category_translation.ingestion_timestamp, 
                    product_category_translation.silver_processed_timestamp, 
                    product_category_translation.data_quality_status, 
                    product_category_translation.source_file_name) 
    FROM
        (SELECT * EXCEPT(products.product_id, product_description_lenght, 
                        product_photos_qty, product_weight_g, product_length_cm, 
                        product_height_cm, product_width_cm, product_name_lenght, 
                        products.data_quality_status, products.ingestion_timestamp, 
                        products.silver_processed_timestamp)  
        FROM 
            (SELECT COUNT(product_id) AS quantity, 
                    order_id, product_id, seller_id, 
                    SUM(price) AS product_subtotal, 
                    SUM(freight_value) AS freight_total
            FROM revenue_operations.silver.order_items
            GROUP BY order_id, product_id, seller_id) AS order_quantity_counts
        LEFT JOIN revenue_operations.silver.products 
        ON order_quantity_counts.product_id = products.product_id) AS products_quantity_category
    LEFT JOIN revenue_operations.silver.product_category_translation
    ON products_quantity_category.product_category_name = product_category_translation.product_category_name) AS product_quantity_category_english)

In [0]:
# Query the temp view to get the data into a DataFrame
fact_order_items = spark.sql("SELECT * FROM fact_order_items")
#display(fact_order_items)
print("Fact order items = ", fact_order_items.count())
print("Order Items =", order_items.count())

In [0]:
%sql
SELECT order_id, product_id, seller_id, COUNT(*)
FROM fact_order_items
GROUP BY order_id, product_id, seller_id
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT *
FROM fact_order_items
WHERE order_id IS NULL OR product_id IS NULL OR seller_id IS NULL

In [0]:
fact_order_items.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.gold.fact_order_items")

#### Dim Customers
- Intended grain = One row represents a unique customer's location, and their purchase information/activity in the store.
- Source Silver tables = revenue_operations.silver.customers, revenue_operations.silver.orders, revenue_operations.silver.payments, revenue_operations.silver.reviews
- Columns selected = customer_unique_id, customer_zip_code_prefix, customer_city, customer_state, first_purchase_date, last_purchase_date, total_orders, total_payment_value, average_order_value, most_recent_customer_id, customer_lifetime_days, repeat_customer_flag
- Calculated fields = first_purchase_date, last_purchase_date, total_orders, total_payment_value, average_order_value, most_recent_customer_id, customer_lifetime_days, repeat_customer_flag, customer_zip_code_prefix, customer_city, customer_state
- Joins used = LEFT JOINs to join fact_orders to customers, on customer_id, and to join most_recent_customer (the ranked table for customer_id based on most recent purchase timestamp), on customer_unique_id
- row count = 96096 ( Less than 99441 rows from customers table because only the most recent customer_id that was related to order_ids were taken)
- duplicate-key validation = No duplicate keys
- null check on key fields = NO null in key fields

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW dim_customers AS (
WITH customer_order_payment AS(
    SELECT  c.customer_id, 
            c.customer_unique_id, 
            c.customer_zip_code_prefix, 
            c.customer_city, 
            c.customer_state,
            o.order_purchase_timestamp, 
            o.order_id, 
            o.total_payment_value,
            o.purchase_date
    FROM revenue_operations.silver.customers c
    LEFT JOIN fact_orders o
    ON c.customer_id = o.customer_id
),
ranked_customer_id AS (
    SELECT *,
           ROW_NUMBER() OVER(PARTITION BY customer_unique_id ORDER BY order_purchase_timestamp DESC) AS rn
    FROM customer_order_payment
),
most_recent_customer AS (
    SELECT customer_unique_id, 
           customer_id AS most_recent_customer_id,
           customer_zip_code_prefix,
           customer_city,
           customer_state
    FROM ranked_customer_id
    WHERE rn = 1
),
customer_order_agg AS (
    SELECT customer_unique_id,
           MIN(purchase_date) AS first_purchase_date,
           MAX(purchase_date) AS last_purchase_date,
           COUNT(DISTINCT order_id) AS total_orders,
           SUM(total_payment_value) AS total_payment_value,
           AVG(total_payment_value) AS average_order_value
    FROM customer_order_payment
    GROUP BY customer_unique_id
)
SELECT agg.*,
       mrc.customer_zip_code_prefix,
       mrc.customer_city,
       mrc.customer_state,
       mrc.most_recent_customer_id,
       DATEDIFF(last_purchase_date, first_purchase_date) AS customer_lifetime_days,
       CASE WHEN total_orders > 1 THEN 'repeat' ELSE 'one-time' END AS repeat_customer_flag
FROM customer_order_agg agg
LEFT JOIN most_recent_customer mrc 
ON agg.customer_unique_id = mrc.customer_unique_id)

In [0]:
dim_customers = spark.sql("SELECT * FROM dim_customers")
display(dim_customers)
dim_customers.count()

In [0]:
customers.count()

In [0]:
%sql
SELECT customer_unique_id, COUNT(*)
FROM dim_customers
GROUP BY customer_unique_id
HAVING COUNT(*) > 1

In [0]:
%sql
SELECT *
FROM dim_customers
WHERE customer_unique_id IS NULL OR most_recent_customer_id IS NULL

In [0]:
dim_customers.write.format("delta").mode("overwrite").saveAsTable("revenue_operations.gold.dim_customers")